# 路线重审：已提交安全汇总的计数复核
不读取题目、答案或教材。主题分配不是语义覆盖证明；历史比较不是新训练结果。

In [ ]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == 'docs': root = root.parent
def read(name): return json.loads((root / 'docs' / name).read_text(encoding='utf-8'))
stage = read('logistics_stage0_evaluation_summary_20260903.safe.json')
diag = read('cpt_targeted_diagnosis_results_20260907.safe.json')
train = read('targeted_book_pilot_prepared_20260908.safe.json')
pairs = read('sft_next_minimal_pairs_20260908.safe.json')
sft = read('book_sft_diagnosis_metrics_20260907.safe.json')
n = stage['inputs']['unique_item_hashes']
profile = diag['manifest']['error_profile']
assert n == sum(r['items'] for r in profile) == 1672
assert sum(r['wrong'] for r in profile) == diag['manifest']['baseline_wrong'] == 301
assert sum(train['train_topics'].values()) == train['train_count'] == 80
assert sum(pairs['accepted_topics'].values()) == pairs['accepted'] == 44
mh = next(r for r in profile if r['category'] == 'material_handling')
assert (mh['items'], mh['wrong']) == (696, 125)
assert diag['cohorts']['cpt_historical_wrong']['evidence_not_auto_verified'] == 292
assert stage['benchmark_quality']['logistika_items_with_269_options'] == 305
assert sft['baseline_correct'] == n - 301 == 1371
assert sft['sft_correct'] - sft['baseline_correct'] == -26
assert sft['transitions']['improved'] - sft['transitions']['regressed'] == -26
nets = {r['category']: r['net'] for r in sft['segments'] if r['dataset'] == 'LogistikaBench'}
assert [nets[k] for k in ['material_handling', 'transport', 'warehousing']] == [-12, -11, 3]
print({'material_error_share': 125/301, 'material_train_share': 6/80, 'material_accepted_share': 2/44, 'large_pool_share_logistika': 305/1446, 'cpt_net_points_vs_step120': (1371-1369)/1672*100, 'proposed_max_local_requests': 96*2*3*3})
print('PASS: denominators and historical evidence reconciled; no new training or inference.')
